# 天气与特征：本轮生成、可用时点及覆盖

本页读取已归档特征审计与字典。以下方法契约解释不同信息组；是否实际入模以本轮 manifest 的特征清单为准，不把尚未启用的信息组说成已完成实验。

In [1]:
ROOT = __import__('pathlib').Path.cwd()
RUN_DIR = ROOT / 'experiments/runs/R0011'

In [2]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown

if "ROOT" not in globals():
    ROOT = Path.cwd()
    while not (ROOT / "scripts/build_features.py").exists():
        if ROOT.parent == ROOT:
            raise RuntimeError("请在 Airline-delay 仓库中运行 Notebook")
        ROOT = ROOT.parent
if "RUN_DIR" not in globals():
    candidates = [p for p in (ROOT / "experiments/runs").glob("R*")
                  if (p / "metrics.csv").exists() and (p / "model_manifest.json").exists()]
    if not candidates:
        raise RuntimeError("请先归档预测实验，或手动设置 RUN_DIR")
    RUN_DIR = max(candidates, key=lambda p: int(p.name[1:]))
TABLES = RUN_DIR / "tables"
FIGURES = RUN_DIR / "figures"
def table(name):
    return pd.read_csv(TABLES / name)
def figure(name):
    display(Image(filename=str(FIGURES / name)))
def receipt(name):
    return json.loads((RUN_DIR / name).read_text(encoding="utf-8"))
display(Markdown(f"**归档运行：{RUN_DIR.name}**。本 Notebook 读取该轮表格与图，不重新拟合模型。"))

**归档运行：R0011**。本 Notebook 读取该轮表格与图，不重新拟合模型。

In [3]:
config = receipt("config.json") if (RUN_DIR / "config.json").exists() else {}
model_manifest = receipt("model_manifest.json")
features = model_manifest.get("features", config.get("features", []))
group_name = model_manifest.get("feature_group", config.get("feature_group", "未登记"))
models = model_manifest.get("models", [])
display(Markdown(f"本轮信息组：**{group_name}**；阶段：**{config.get('stage', '未登记')}**；模型：**{', '.join(models)}**。"))
display(pd.DataFrame([{"development_only": model_manifest.get("development_only"),
                       "parent_run_id": config.get("parent_run_id"),
                       "change": config.get("change"), "include_baselines": model_manifest.get("include_baselines")}]))

本轮信息组：**G3**；阶段：**P3**；模型：**training_mean, training_median, carry_forward, g3_linear**。

,development_only,parent_run_id,change,include_baselines
0,True,R0009,Add historical airport departures and upcoming...,True


## 1. 实际特征及生成审计

目标与标签时刻是评价元数据，不是预测变量。只在训练折拟合插补、类别处理和缩放；缺失不能按无延误、晴天或已知零填充。

In [4]:
display(pd.DataFrame({"实际入模字段": features}))
if (RUN_DIR / "feature_audit.json").exists():
    audits = receipt("feature_audit.json")
    for section, values in audits.items():
        display(Markdown(f"**{section}**"))
        display(pd.json_normalize(values) if isinstance(values, dict) else values)
else:
    display(Markdown("本轮未归档 feature_audit.json。"))
dictionary_path = RUN_DIR / "feature_dictionary.csv"
if dictionary_path.exists():
    display(pd.read_csv(dictionary_path))
else:
    display(Markdown("本轮未归档特征字典。"))

,实际入模字段
0,airport_id
1,route
2,downstream_airline
3,aircraft_family
4,downstream_distance
5,downstream_crs_elapsed_time
6,downstream_day_of_week
7,downstream_sched_hour
8,scheduled_turnaround_minutes
9,upstream_arr_delay_minutes


**feature_audit**

,development_only,categorical_features,numeric_features,outcome_rule,feature_group,features,counts.input_connections,counts.outside_prediction_utc_scope,counts.scoped_connections,counts.missing_or_nonfinite_target,counts.negative_target,counts.missing_label_time,counts.prediction_not_before_departure,counts.missing_or_invalid_upstream_delay,counts.eligible_connections
0,True,"[airport_id, route, downstream_airline, aircra...","[downstream_distance, downstream_crs_elapsed_t...",finite target >= 0 and prediction_time < label...,G3,"[airport_id, route, downstream_airline, aircra...",2821852,560077,2261775,0,0,0,0,0,2261775


**history_audit**

,history_window,include_current_upstream,source_start_utc,current_upstream_rule,event_quality_rule,ignored_future_qa_columns,quality_exclusions_are_sequential,missingness_rule,left_truncation_rule,carry_in_limitation,...,counts.duplicate_flight_id_rows,counts.unique_history_flight_ids,counts.unique_history_tails,counts.same_tail_arrival_tie_rows_retained,queries.rows,queries.missing_tail,queries.known_tail_zero_history,queries.history_window_incomplete,queries.current_upstream_added,queries.events_present
0,"[prediction_time - 24 hours, prediction_time)",False,2026-04-01T11:00:00+00:00,Excluded by default per plan: previously compl...,"unique flight_id, present tail, normal_operati...","[tail_schedule_incomplete, schedule_tie, actua...",True,missing tail: n and mean missing; known tail w...,"Known-left-truncation indicator, not a complet...",Flights with prior-local-month departure dates...,...,0,2395389,6069,12,2261775,0,53090,18121,0,2208685


**aircraft_audit**

,inventory_year,join_keys,rows,family_matched,variant_matched,unknown_preserved,limitation,linear_rule
0,2024,"[downstream_airline, tail_number]",2261775,2002907,2002907,True,"Static B43 inventory snapshot, not the histori...","Use family OR variant, never both nested categ..."


**airport_operations_audit**

,historical_window,scheduled_window,historical_departures,planned_rows,historical_quality,plan_assumption,no_event_mean,source_start_rule,left_truncation_rule,carry_in_limitation,left_window_incomplete_queries
0,"[prediction_time-60min, prediction_time)","[prediction_time, prediction_time+60min)",2409279,3057590,Departure clock/delay consistency and unique s...,Archived BTS scheduled times are treated as kn...,"NA; count is zero, training-only imputation",Per-airport source local-month midnight conver...,Indicator of known source-left truncation; zer...,Prior-local-month flights departing late are a...,0


,feature,group,meaning,unit,source,generation_rule,available_at,missing_handling,role
0,airport_id,G0,Stable connection/downstream origin AirportID,category,BTS OriginAirportID,Use stable ID; aliases do not create extra air...,planned before prediction (snapshot assumption),constant missing category fitted in training,predictor
1,route,G0,Downstream origin to destination,category,BTS OriginAirportID/DestAirportID,originID -> destID,planned before prediction (snapshot assumption),constant missing category fitted in training,predictor
2,downstream_airline,G0,Downstream reporting airline,category,BTS Reporting_Airline,Use planned reporting airline identifier,planned before prediction (snapshot assumption),constant missing category fitted in training,predictor
3,downstream_distance,G0,Downstream route distance,miles,BTS Distance,Copy scheduled route distance,planned before prediction,training median plus missing indicator,predictor
4,downstream_crs_elapsed_time,G0,Downstream scheduled elapsed time,minutes,BTS CRSElapsedTime,Copy scheduled duration,planned before prediction,training median plus missing indicator,predictor
5,downstream_day_of_week,G0,Downstream local scheduled weekday,1=Monday..7=Sunday,BTS DayOfWeek,Original planned local date weekday; numerical...,planned before prediction,training median plus missing indicator,predictor
6,downstream_sched_hour,G0,Downstream local scheduled departure hour,hour 0..23,BTS CRSDepTime,HHMM modulo 24h; 2400 -> hour0; numerical P0 e...,planned before prediction,training median plus missing indicator,predictor
7,scheduled_turnaround_minutes,G0,Scheduled upstream arrival to downstream depar...,minutes,reconstructed schedule UTC,downstream scheduled departure minus upstream ...,planned before prediction,training median plus missing indicator,predictor
8,upstream_arr_delay_minutes,baseline/G1,Known nonnegative upstream arrival delay,minutes,BTS ArrDelayMinutes,Known at prediction time; not a G0 predictor,upstream actual arrival,must be observed for carry-forward cohort,baseline or later predictor
9,prediction_time,metadata,Prediction timestamp,UTC datetime,reconstructed actual upstream arrival,scheduled arrival plus raw ArrDelay,prediction instant,exclude absent timestamp,split metadata


## 2. 航班链、飞机映射及机场运行

航班链历史默认使用预测前 24 小时已完成的航段，不重复计入当前上游；其已知到达延误单独作为字段。已知尾号没有观察到有效历史时，计数为零、均值缺失；未知尾号保留缺失。history_window_incomplete 是**已知左截断指示**，不是 flag=0 时的完整覆盖保证。源数据按当地 4 月起飞日期取样，前月起飞而延迟到达/出发的 carry-in 航班可能缺失。

B43 是静态参考清单，以航司＋尾号连接，不能当作逐预测时点的历史机队快照。family 与 variant 单独比较，含截距线性模型不同时冗余展开二者。

机场历史窗口是 \( [t-60\mathrm{min},t) \)，只计已实际出发事件；计划负荷窗口是 \( [t,t+60\mathrm{min}) \)。最终取消状态不能用于事后删去计划负荷。计划快照事前已知是研究假设，仓库没有逐时计划修订版本。

## 3. 天气时点契约

METAR 假设观测后 5 分钟可用，观测年龄最多 120 分钟；这是建模假设，不是源数据提供的接收时间。TAF 先选预测时点已可用的最新公告，再匹配查询时刻 max(下游计划出发, prediction_time)，并处理有效期、修订、取消/NIL 及临时/概率段。NOAA 仅作独立补缺分支，保留质量码、单位核对和逐值来源。缺天气保留样本及缺失标记，覆盖对照须核对评估 ID。

## 4. 实际缺失、映射与覆盖表

以下仅展示本轮确实归档的表。没有对应表不代表该信息源覆盖完整，也不代表该信息组已经启用。

In [5]:
coverage_paths = sorted({p for folder in [RUN_DIR, TABLES] if folder.exists()
                         for p in folder.glob("*.csv")
                         if any(word in p.stem.lower() for word in ["missing", "coverage", "match_rate", "unmatched"])})
for path in coverage_paths:
    display(Markdown(f"**{path.relative_to(RUN_DIR)}**"))
    data = pd.read_csv(path)
    display(Markdown(f"归档表共 {len(data):,} 行；下方展示前 50 行。"))
    display(data.head(50))
if not coverage_paths:
    display(Markdown("本轮没有单独归档缺失/覆盖表；请结合上方审计阅读。"))

**tables\eda_feature_missing.csv**

归档表共 8 行；下方展示前 50 行。

,feature,missing_n,missing_share
0,airport_id,0,0.0
1,route,0,0.0
2,downstream_airline,0,0.0
3,downstream_distance,0,0.0
4,downstream_crs_elapsed_time,0,0.0
5,downstream_day_of_week,0,0.0
6,downstream_sched_hour,0,0.0
7,scheduled_turnaround_minutes,0,0.0


**tables\feature_missing.csv**

归档表共 18 行；下方展示前 50 行。

,feature,missing_n,rows,missing_share
0,airport_id,0,2261775,0.000000
1,route,0,2261775,0.000000
2,downstream_airline,0,2261775,0.000000
3,aircraft_family,258868,2261775,0.114453
4,downstream_distance,0,2261775,0.000000
5,downstream_crs_elapsed_time,0,2261775,0.000000
6,downstream_day_of_week,0,2261775,0.000000
7,downstream_sched_hour,0,2261775,0.000000
8,scheduled_turnaround_minutes,0,2261775,0.000000
9,upstream_arr_delay_minutes,0,2261775,0.000000
